# **AutoML com AutoKeras**
- O Automated Machine Learning é também conhecido como AutoML;
- Ferramentas e serviços que abstraem os detalhes e conhecimentos necessários para se realizar ML/DL;
- Automatiza-se determinadas tarefas necessárias no desenvolvimento de algum modelo;
- Geralmente, cobrem etapas de normalização de dados e engenharia de “features”; treinamento de modelos de diferentes tipos e com diferentes hiperparâmetros; avaliação e comparação de resultados;
- Algumas ferramentas: H2O, AutoKeras e TPOT;

- Vantagens:  
  - Necessário conhecimento superficial de ML/DL para se utilizar as ferramentas;
  - Diminui o tempo para se ter resultados iniciais;
  - Pode-se utilizar em diferentes tipos de dados e problemas (baseline);
  - Algumas ferramentas possibilitam customizar modelos/layers;
- Desvantagens:
  - Customização que pode ser limitada ou dificultada;
  - Pode-se não obter resultado bom o bastante para o problema, precisando da intervenção de especialistas;



In [1]:
!pip install "keras-nlp<0.9.0" "keras-tuner<1.4.0" "autokeras<3.0" "tensorflow<2.16.0" "keras<3.0" -q

In [2]:
#!pip install autokeras==1.0.19 -q

In [3]:
!pip install ipywidgets

In [4]:
import tensorflow
from tensorflow import keras
from keras import preprocessing
from autokeras import StructuredDataClassifier

import os
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

Using TensorFlow backend


In [5]:
%%time
!rm -rf lectures-cdas-2023
!git clone https://github.com/renansantosmendes/lectures-cdas-2023.git

Cloning into 'lectures-cdas-2023'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 0), reused 12 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 174.34 KiB | 3.71 MiB/s, done.
CPU times: user 10.8 ms, sys: 16.4 ms, total: 27.3 ms
Wall time: 1.26 s


In [6]:
data = pd.read_csv(os.path.join('lectures-cdas-2023', 'fetal_health.csv'))

In [7]:
X=data.drop(["fetal_health"], axis=1)
y=data["fetal_health"]

columns_names = list(X.columns)
X_df = sklearn.preprocessing.StandardScaler().fit_transform(X)
X_df = pd.DataFrame(X_df, columns=columns_names)

X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.3, random_state=42)
y_train=keras.utils.to_categorical(y_train)[:,1:]
y_test=keras.utils.to_categorical(y_test)[:,1:]

In [8]:
# Adicionado para adaptar notebook

# Converte as tabelas do Pandas para matrizes do NumPy
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()

In [9]:
%%time
clf = StructuredDataClassifier(overwrite=True, max_trials=15)
clf.fit(x=X_train, y=y_train, epochs=10)

Trial 15 Complete [00h 00m 01s]
val_accuracy: 0.9227941036224365

Best val_accuracy So Far: 0.9301470518112183
Total elapsed time: 00h 00m 23s
INFO:tensorflow:Oracle triggered exit


INFO:tensorflow:Oracle triggered exit


Epoch 1/10
47/47 [==============================] - 0s 758us/step - loss: 0.5748 - accuracy: 0.7970
Epoch 2/10
47/47 [==============================] - 0s 657us/step - loss: 0.3441 - accuracy: 0.8656
Epoch 3/10
47/47 [==============================] - 0s 543us/step - loss: 0.2805 - accuracy: 0.8925
Epoch 4/10
47/47 [==============================] - 0s 553us/step - loss: 0.2432 - accuracy: 0.9113
Epoch 5/10
47/47 [==============================] - 0s 539us/step - loss: 0.2131 - accuracy: 0.9227
Epoch 6/10
47/47 [==============================] - 0s 545us/step - loss: 0.1879 - accuracy: 0.9382
Epoch 7/10
47/47 [==============================] - 0s 531us/step - loss: 0.1657 - accuracy: 0.9489
Epoch 8/10
47/47 [==============================] - 0s 536us/step - loss: 0.1443 - accuracy: 0.9590
Epoch 9/10
47/47 [==============================] - 0s 545us/step - loss: 0.1236 - accuracy: 0.9698
Epoch 10/10
47/47 [==============================] - 0s 550us/step - loss: 0.1055 - accuracy: 0.9772

INFO:tensorflow:Assets written to: ./structured_data_classifier/best_model/assets


CPU times: user 28.8 s, sys: 5.52 s, total: 34.3 s
Wall time: 25.6 s


In [10]:
model = clf.export_model()

In [13]:
model

In [11]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 21)]              0         
                                                                 
 multi_category_encoding (M  (None, 21)                0         
 ultiCategoryEncoding)                                           
                                                                 
 normalization (Normalizati  (None, 21)                43        
 on)                                                             
                                                                 
 dense (Dense)               (None, 256)               5632      
                                                                 
 batch_normalization (Batch  (None, 256)               1024      
 Normalization)                                                  
                                                             

In [12]:
model.evaluate(X_test, y_test)

20/20 [==============================] - 0s 583us/step - loss: 0.2606 - accuracy: 0.8934


[0.26058244705200195, 0.8934169411659241]